<a href="https://colab.research.google.com/github/tuan314159265/parking-occupancy/blob/main/notebooks/01_pklot_run_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nhận diện trạng thái ô đỗ xe — PKLot


## 1. Bài toán và mục tiêu


### Bối cảnh và mục tiêu

Trong bãi xe ngoài trời, việc xác định vị trí còn trống từ camera có thể hỗ trợ người quản lý theo dõi mức sử dụng bãi mà không cần kiểm tra từng ô trực tiếp. Bài toán gặp khó khăn khi xe ở xa camera, bóng đổ che đường kẻ, ánh sáng thay đổi hoặc mặt đường bị ảnh hưởng bởi mưa.

Đề tài xây dựng hệ thống nhận diện trạng thái ô đỗ và đếm số chỗ trống từ ảnh màu toàn cảnh. Hệ thống được thiết kế để so sánh ba phương pháp: phân loại ảnh từng ô bằng HOG kết hợp SVM, phát hiện xe bằng YOLO11n trên toàn ảnh và phát hiện xe bằng cùng detector trên các ROI/tile. Việc so sánh xem xét chất lượng nhận diện, sai số đếm và chi phí xử lý; lợi ích của chia tile cần được kiểm chứng bằng số liệu.

### Đầu vào và đầu ra

Đầu vào gồm ảnh màu toàn cảnh từ camera cố định và cấu hình các ô đỗ tương ứng với ảnh. Mỗi ô có mã định danh và polygon biểu diễn vùng ô trong hệ tọa độ ảnh gốc. Ngày thu nhận, camera và thời tiết được giữ lại để chia dữ liệu và phân tích kết quả. Cấu hình phương pháp gồm checkpoint, kích thước đầu vào, các quy tắc ghép xe với ô và ngưỡng đã lựa chọn trên validation.

Đầu ra của mỗi ảnh gồm mã ô, vị trí vùng ô, trạng thái có xe/trống/chưa xác định và điểm có xe khi phương pháp cung cấp được. Tổng số chỗ trống được tính từ những ô được xác định là trống; số ô chưa xác định được báo riêng. Kết quả được trực quan hóa trên ảnh gốc và lưu dưới dạng CSV/JSON để đánh giá và đối chiếu.

### Phân biệt phát hiện xe và nhận diện trạng thái ô

Phát hiện xe trả về vị trí hộp bao của các đối tượng xe và độ tin cậy. Nhận diện trạng thái ô trả lời một vùng ô đỗ đã biết đang có xe hay trống. Với phương pháp dựa trên detector, cần thêm bước ánh xạ hộp xe về ảnh gốc và ghép hình học xe–ô trước khi suy ra trạng thái. Với HOG/SVM, ảnh của từng ô được phân loại trực tiếp theo nhãn trạng thái.

Polygon hoặc hình chữ nhật xoay trong XML PKLot biểu diễn vùng ô đỗ; chúng không xác định hộp bao của chiếc xe. Vì vậy, nhãn ô không được chuyển thành nhãn hộp xe để huấn luyện detector hoặc tính mAP phát hiện xe. Phần đánh giá chính sử dụng trạng thái từng ô, phép đếm và thời gian trên ảnh gốc.

### Phạm vi và giả định

Phạm vi chính là bãi xe ngoài trời với camera cố định và các vùng ô đỗ đã được cung cấp. Khi vị trí hoặc góc nhìn camera thay đổi, cấu hình vùng ô phải được cập nhật hoặc xác nhận lại. Hệ thống không giả định một cấu hình polygon duy nhất cho mọi ảnh nếu dữ liệu ghi nhận nhiều biến thể hình học.

Canny và Hough được dùng trên nhánh ảnh xám để phân tích biên, đường kẻ và kiểm tra chất lượng hình học. Nhánh detector sử dụng ảnh màu; việc không tìm thấy đường kẻ không được dùng riêng để kết luận ô trống. Tự động đề xuất vị trí ô đỗ là hướng mở rộng, chưa thuộc phạm vi thực nghiệm chính. Kết quả sẽ được phân tích theo thời tiết và camera; thí nghiệm chuyển góc nhìn trong cùng bãi được trình bày riêng.


## 2. Môi trường và cấu hình


In [31]:
OWNER = "tuan314159265"
REPO = "parking-occupancy"
REF = "main"
MODULE_PATH = "parking_module.py"

from pathlib import Path
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "parking-occupancy"  # Thư mục CÁ NHÂN để lưu kết quả/module.
DOWNLOAD_DATA = True
FORCE_AUDIT = False  # True nếu dataset đã thay đổi hoặc muốn kiểm tra lại toàn bộ.

# Để trống: tự tìm/tải dataset trong thư mục cá nhân.
# Dùng chung: thêm shortcut Drive, rồi điền thư mục chứa camera UFPR04/UFPR05/PUCPR.
SHARED_DATASET_DIR = ""  # Ví dụ /content/drive/MyDrive/PKLot-shared/data/raw/PKLot
# Không bắt buộc: thư mục outputs/full của chủ dataset (chỉ đọc).
SHARED_AUDIT_DIR = ""  # Ví dụ /content/drive/MyDrive/PKLot-shared/outputs/full

# Bộ manifest dùng chung, đã khóa trên GitHub.
PROJECT_DATASET_VERSION = "pklot_v1"
PROTOCOL = "within_camera"  # Hoặc transfer_ufpr04_to_ufpr05; phải huấn luyện lại trên split của giao thức đó.


### Kết nối Google Drive

Trong cửa sổ cấp quyền Google, chọn tài khoản Drive của bạn, đánh dấu **tất cả các quyền được yêu cầu** (hoặc **Select all**) rồi bấm **Continue**. Bỏ sót một quyền có thể gây lỗi `credential propagation was unsuccessful`. Không chỉ chọn riêng quyền đọc/ghi Drive.

Nếu không thấy cửa sổ, kiểm tra popup bị chặn. Nếu vẫn lỗi dù đã cấp đủ quyền, thử Runtime → Disconnect and delete runtime rồi kết nối lại. Thao tác này xóa dữ liệu tạm của runtime, giữ file trên Drive. Nếu dùng local runtime, đặt USE_GOOGLE_DRIVE=False.

Nguồn: https://github.com/googlecolab/colabtools/issues/6089


In [32]:
import os
from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        from google.colab._message import MessageError
    except ImportError as exc:
        raise RuntimeError(
            "Kết nối bằng runtime Google Colab để dùng Drive. "
            "Nếu chạy local, đặt USE_GOOGLE_DRIVE=False."
        ) from exc

    mountpoint = Path("/content/drive")
    def drive_ready():
        return os.path.ismount(mountpoint) and (mountpoint / "MyDrive").is_dir()

    if not drive_ready():
        for attempt in range(2):
            try:
                print("Kết nối Google Drive: chọn tài khoản, chọn TẤT CẢ quyền được yêu cầu rồi Continue.")
                drive.mount(str(mountpoint), force_remount=(attempt == 1), timeout_ms=120000)
                if not drive_ready():
                    raise RuntimeError("Drive chưa có thư mục MyDrive sau khi mount.")
                break
            except MessageError as exc:
                raise RuntimeError(
                    "Colab chưa nhận được thông tin xác thực Google (MessageError). "
                    "Chạy lại cell này; trong hộp thoại đăng nhập, chọn đúng tài khoản "
                    "và đánh dấu tất cả quyền Drive/Colab được yêu cầu trước khi Continue. "
                    "Nếu không thấy hộp thoại, kiểm tra trình duyệt có chặn popup. "
                    "Nếu vẫn lỗi, dùng Runtime > Disconnect and delete runtime rồi kết nối lại. "
                    "Chưa tải dataset và chưa chuyển sang bộ nhớ tạm. "
                    f"Chi tiết gốc: {exc}"
                ) from exc
            except (ValueError, OSError, RuntimeError) as exc:
                if attempt == 0:
                    print("Kết nối chưa thành công; thử lại bằng force_remount=True.")
                else:
                    raise RuntimeError(
                        "Không kết nối được Google Drive sau 2 lần thử. "
                        "Nếu dùng local runtime, chuyển sang runtime Google Colab. "
                        "Trên Colab: chọn Runtime > Disconnect and delete runtime, "
                        "kết nối lại rồi Run All và cấp quyền Drive. "
                        "Để chạy tạm không dùng Drive, đặt USE_GOOGLE_DRIVE=False; "
                        "khi đó dữ liệu chỉ nằm trong phiên Colab. "
                        "Chưa tải dataset vì Drive chưa sẵn sàng."
                    ) from exc
    WORK_DIR = mountpoint / "MyDrive" / DRIVE_PROJECT_DIR
    print("Dữ liệu và kết quả sẽ được lưu trên Google Drive.")
else:
    WORK_DIR = Path("/content/parking-occupancy") if Path("/content").is_dir() else Path.cwd()
    print("Chế độ không dùng Drive. Trên Colab, dữ liệu sẽ mất khi runtime bị xóa.")

RAW_DIR = WORK_DIR / "data/raw"
DATASET_DIR = Path(SHARED_DATASET_DIR) if SHARED_DATASET_DIR else RAW_DIR / "PKLot"
OUTPUT_DIR = WORK_DIR / "outputs/full"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu:", WORK_DIR)


Dữ liệu và kết quả sẽ được lưu trên Google Drive.
Thư mục lưu: /content/drive/MyDrive/parking-occupancy


In [33]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10,<13"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'Pillow>=10,<13'], returncode=0)

In [34]:
import hashlib, importlib.util, json, sys, time, urllib.request
from urllib.parse import quote
assert OWNER != "YOUR_GITHUB_USERNAME", "Điền OWNER, REPO và REF ở cell cấu hình"
WORK_DIR.mkdir(parents=True, exist_ok=True)
url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/"
       f"{quote(REPO, safe='')}/{quote(REF, safe='')}/{quote(MODULE_PATH, safe='/')}")
# Tải lại module mỗi lần Run All để lấy đúng REF đã cấu hình.
request = urllib.request.Request(
    f"{url}?refresh={time.time_ns()}", headers={"Cache-Control": "no-cache"}
)
with urllib.request.urlopen(request, timeout=60) as response:
    code = response.read()
compile(code, MODULE_PATH, "exec")
module_file = WORK_DIR / "parking_module.py"
module_file.write_bytes(code)
spec = importlib.util.spec_from_file_location("parking_module", module_file)
parking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parking
spec.loader.exec_module(parking)
provenance = {"url": url, "ref": REF, "sha256": hashlib.sha256(code).hexdigest()}
(WORK_DIR / "module_source.json").write_text(json.dumps(provenance, indent=2))
print("Loaded module:", provenance)


Loaded module: {'url': 'https://raw.githubusercontent.com/tuan314159265/parking-occupancy/main/parking_module.py', 'ref': 'main', 'sha256': '1ec46218d76911f3043f3f3f3e33210ea99270f03e6b56d295c1829f8290cfa4'}


### Giao diện cho các bước tiếp theo

Gọi trực tiếp `parking.VehicleDetector`, `parking.OccupancyPrediction`, `parking.occupancy_metrics` khi triển khai detector và ghép ô. Polygon XML là vùng ô đỗ, không phải hộp xe. Chưa triển khai mô hình trong notebook này.


## 3. Dữ liệu và tiền xử lý


### 3.1. Tải và kiểm tra dữ liệu PKLot


#### Nguồn dữ liệu và tổ chức PKLot

PKLot được lựa chọn vì cung cấp ảnh bãi xe toàn cảnh cùng thông tin vùng ô và trạng thái chiếm dụng. Theo công bố của bộ dữ liệu, PKLot có 12.417 ảnh màu độ phân giải 1280 × 720 pixel, thu nhận tại hai bãi xe ngoài trời trong các điều kiện nắng, nhiều mây và mưa. Nguồn dữ liệu và công trình giới thiệu PKLot được dẫn tại [trang PKLot của UFPR](https://web.inf.ufpr.br/vri/databases/parking-lot-database/) và [tài liệu giới thiệu của tác giả](https://www.inf.ufpr.br/lesoliveira/download/pklot-readme.pdf).

Archive sử dụng trong dự án có ba góc nhìn PUCPR, UFPR04 và UFPR05. UFPR04 và UFPR05 quan sát cùng bãi UFPR từ hai góc nhìn; PUCPR thuộc bãi khác. Ảnh được tổ chức theo camera, thời tiết và ngày. Dự án sử dụng ảnh toàn cảnh và XML để giữ bối cảnh cho detector; bộ ảnh từng ô đã cắt sẵn không thay thế đầu vào toàn cảnh.

#### Nhãn ô đỗ và cấu hình hình học

Mỗi phần tử space trong XML có mã ô, nhãn occupied và contour của vùng ô. Nhãn 1 biểu thị có xe, nhãn 0 biểu thị trống. Nhãn thiếu hoặc không hợp lệ không được suy đoán thành nhãn trống. Trạng thái chưa xác định là đầu ra mà hệ thống có thể sử dụng khi chưa đủ cơ sở quyết định; đó không phải lớp ground truth bổ sung của tập con đã chốt.

Tọa độ polygon được giữ trong ảnh gốc, gốc ở góc trên trái, trục x hướng sang phải và trục y hướng xuống dưới. Bộ JSON hình học của tập con có 5 layout: PUCPR có 1, UFPR04 có 3 và UFPR05 có 1. Mỗi frame liên kết đến layout bằng geometry_id, còn nhãn trạng thái được lưu riêng theo mã ô. Cách biểu diễn này giữ được các biến thể chú thích thay vì ép mọi ảnh cùng camera dùng một polygon cố định.

#### Kiểm tra chất lượng và bảo toàn metadata

Với archive local đã kiểm tra bằng loader v2, dự án ghi nhận 12.416 XML và 1 ảnh thiếu XML. Trong số các frame có XML, 11.638 frame hợp lệ được giữ lại; 778 frame bị loại do nhãn hoặc mã ô không hợp lệ. Số lượng này là kết quả kiểm tra của dự án, khác với số ảnh được công bố cho toàn bộ PKLot. Danh sách lỗi được lưu để truy vết; không tự gán nhãn thay thế hoặc bỏ âm thầm ô lỗi trong một frame.

Loader hỗ trợ cả hai cách viết Point/point trong contour. Ngày thu nhận chuẩn được lấy từ timestamp hợp lệ trong tên file; ngày thư mục gốc được giữ trong directory_date và nguồn ngày được ghi trong date_source. Có 195 frame hợp lệ có ngày trong tên file khác ngày thư mục, bao gồm thư mục có tháng không hợp lệ. Dữ liệu gốc không bị đổi tên. Thời tiết Cloudy được chuẩn hóa thành overcast, đồng thời đường dẫn gốc vẫn được lưu.

Bảng dưới cho biết số frame hợp lệ trước khi chọn tập con thực nghiệm.

| Camera | Bãi xe | Frame hợp lệ |
| --- | --- | --- |
| PUCPR | PUCPR | 4.077 |
| UFPR04 | UFPR | 3.547 |
| UFPR05 | UFPR | 4.014 |
| Tổng |  | 11.638 |


#### Dùng lại dataset đã có

Có dataset XML: bỏ qua tải/giải nén, kể cả thiếu provenance.json. Có audit tương thích: dùng lại manifest và kiểm tra nhanh một mẫu, bỏ qua audit toàn bộ. Thiếu cache: audit một lần và lưu vào thư mục cá nhân. FORCE_AUDIT=True để quét lại sau khi dữ liệu thay đổi. Cache không chứng minh dataset chưa thay đổi; chỉ dùng lại khi bộ dữ liệu được giữ nguyên.

Với dữ liệu chung, điền SHARED_DATASET_DIR và tùy chọn SHARED_AUDIT_DIR. Mã chỉ đọc các thư mục này; không tải hay giải nén vào đó. Nếu đường dẫn chung sai thì dừng để bạn sửa. DRIVE_PROJECT_DIR phải là thư mục cá nhân; không trỏ nó vào shortcut chung. Archive khoảng 4,9 GB; lần đầu cần khoảng 10–15 GB trống nếu tự tải.


In [35]:
summary = parking.ensure_dataset(
    raw=RAW_DIR,
    output=OUTPUT_DIR,
    dataset_root=DATASET_DIR if SHARED_DATASET_DIR else None,
    cached_audit_dir=SHARED_AUDIT_DIR or None,
    download=DOWNLOAD_DATA,
    force_audit=FORCE_AUDIT,
)
print(json.dumps(summary, indent=2))
assert summary["valid_frames"] > 0


Existing dataset: /content/drive/MyDrive/parking-occupancy/data/raw/PKLot. Skip download and extraction.
No compatible audit cache or FORCE_AUDIT=True; audit existing dataset once.
{
  "audit_version": 2,
  "root": "/content/drive/MyDrive/parking-occupancy/data/raw/PKLot",
  "seed": 42,
  "xml_count": 12416,
  "valid_frames": 11638,
  "rejected_frames": 778,
  "images_without_xml": 1,
  "counts": {
    "camera/PUCPR": 4077,
    "weather/overcast": 4097,
    "split/train": 9080,
    "vacant": 348315,
    "occupied": 319261,
    "date_directory_mismatch": 195,
    "weather/rainy": 1206,
    "split/val": 1626,
    "weather/sunny": 6335,
    "split/test": 932,
    "camera/UFPR04": 3547,
    "camera/UFPR05": 4014
  }
}


In [36]:
# Đọc một mẫu hợp lệ trực tiếp bằng module lấy từ GitHub.
with (OUTPUT_DIR / "valid_samples.jsonl").open() as stream:
    sample = json.loads(next(stream))
frame = parking.read_frame(DATASET_DIR / sample["xml_path"], DATASET_DIR)
print("Sample:", frame.sample_id)
print("Camera / weather / date:", frame.camera, frame.weather, frame.date)
print("Parking spaces:", len(frame.spaces))
print("Outputs:", OUTPUT_DIR)


Sample: PUCPR/Cloudy/2012-09-12/2012-09-12_06_05_16
Camera / weather / date: PUCPR overcast 2012-09-12
Parking spaces: 100
Outputs: /content/drive/MyDrive/parking-occupancy/outputs/full


### 3.2. Chia tập dữ liệu và cấu hình vùng ô đỗ

#### Tập con dùng chung và ngân sách xử lý

Để phù hợp với ngân sách Colab và giữ so sánh công bằng, dự án cố định tập con pklot_v1 dùng chung cho mọi phương pháp. Seed là 42; mỗi cặp camera/thời tiết có giới hạn tối đa 140 frame train, 30 validation và 30 test. Việc chọn frame luân phiên theo ngày và dùng thứ tự hash xác định; không lặp ảnh để đạt số lượng mong muốn.

Giao thức chính có tối đa 1.800 ảnh và thực tế chọn được 1.722 ảnh. Hai nhóm train UFPR04/rainy và UFPR05/rainy thiếu lần lượt 22 và 56 ảnh so với giới hạn; không lấy bù từ validation/test. Tập con ưu tiên khả năng so sánh giữa camera/thời tiết, nên phân bố này không đại diện tỉ lệ thời tiết tự nhiên của toàn bộ PKLot.

#### Chia theo nhóm ngày và kiểm thử chuyển camera

Train, validation và test được chia theo ngày thu nhận, không chia ngẫu nhiên từng frame hoặc từng crop ô. Một ngày chỉ thuộc một split trong mỗi giao thức, kể cả khi ngày đó có nhiều thời tiết hoặc camera; tất cả ô của một frame đi cùng split. Quy tắc này hạn chế việc ảnh gần nhau theo thời gian, chứa cùng xe đỗ, xuất hiện đồng thời trong tập học và tập kiểm thử.

Giao thức within_camera đưa cả ba camera vào mỗi split và phân chia nhóm ngày xấp xỉ 70/15/15. Phương án chia được chọn trong 256 ứng viên xác định bởi seed, dựa vào số lượng và metadata camera/thời tiết, không dựa vào dự đoán hoặc điểm số test. Sau khi áp dụng giới hạn ảnh, số frame thực tế được trình bày trong bảng; tỉ lệ nhóm ngày không được hiểu là tỉ lệ ảnh chính xác.

Giao thức transfer_ufpr04_to_ufpr05 dùng UFPR04 cho train/validation và giữ UFPR05 cho test. Mọi ngày của camera đích được loại khỏi nguồn trước khi chia ngày nguồn xấp xỉ 80/20. Đây là kiểm thử góc nhìn mới trong cùng bãi UFPR, không phải kiểm thử chuyển sang một bãi xe độc lập. Hai giao thức là hai thí nghiệm riêng, không gộp manifest hoặc dùng model đã học camera đích để báo kết quả chuyển camera.

#### Vai trò của các split và khả năng tái lập

Train được sử dụng cho các thành phần cần học, như HOG/SVM. Validation được dùng để chọn tham số, padding, overlap và ngưỡng suy trạng thái. Với detector pretrained, checkpoint và cấu hình được chốt trước khi đánh giá test. Sau khi khóa lựa chọn, tất cả phương pháp chạy trên cùng tập test của giao thức tương ứng; không chọn lại tham số bằng kết quả test.

Các manifest train/val/test lưu đường dẫn tương đối, camera, bãi xe, thời tiết, ngày, mã layout và nhãn từng ô. File split_lock.json ghi cấu hình, danh sách ngày và SHA256 các artifact; notebook tải cùng bộ metadata từ GitHub thay vì tự chia lại trên mỗi máy. Bộ kiểm tra xác nhận không giao ngày giữa các split và chỉ có camera đích trong test của thí nghiệm chuyển camera. Bộ pklot_v1 là giao thức của dự án, không phải bản sao split chính thức của PKLot.

| Giao thức | Train | Validation | Test |
| --- | --- | --- | --- |
| Trong camera | 1.182 | 270 | 270 |
| Chuyển UFPR04 → UFPR05 | 420 | 90 | 90 |

Các phương pháp dùng chung `TRAIN_FRAMES`, `VAL_FRAMES`, `TEST_FRAMES` và `SLOT_GEOMETRIES` từ cell dưới.


In [37]:
PROJECT_SPLIT_DIR = WORK_DIR / "splits" / PROJECT_DATASET_VERSION
project_data = parking.fetch_project_splits(
    OWNER, REPO, REF, PROJECT_SPLIT_DIR, dataset_version=PROJECT_DATASET_VERSION
)
SLOT_GEOMETRIES = project_data["geometries"]
SPLIT_LOCK = project_data["lock"]
MANIFEST_DIR = PROJECT_SPLIT_DIR / PROTOCOL
TRAIN_FRAMES = project_data["protocols"][PROTOCOL]["train"]
VAL_FRAMES = project_data["protocols"][PROTOCOL]["val"]
TEST_FRAMES = project_data["protocols"][PROTOCOL]["test"]
print("Dataset / protocol:", PROJECT_DATASET_VERSION, PROTOCOL)
print("Train / validation / test:", len(TRAIN_FRAMES), len(VAL_FRAMES), len(TEST_FRAMES))
print("Geometry layouts:", len(SLOT_GEOMETRIES))
print("Manifest directory:", MANIFEST_DIR)
# Kiểm tra một frame thật; các phương pháp đọc frame theo cùng manifest này.
example_frame = parking.read_project_frame(TRAIN_FRAMES[0], DATASET_DIR)
print("Verified sample:", example_frame.sample_id)


Dataset / protocol: pklot_v1 within_camera
Train / validation / test: 1182 270 270
Geometry layouts: 5
Manifest directory: /content/drive/MyDrive/parking-occupancy/splits/pklot_v1/within_camera
Verified sample: PUCPR/Cloudy/2012-09-12/2012-09-12_06_10_30


## 4. Các phương pháp nhận diện


<!-- TODO -->


### 4.1. HOG và SVM


In [38]:
# TODO


### 4.2. Phân tích hình học bằng Canny và Hough


In [39]:
# TODO


### 4.3. ROI và chia tile


In [40]:
# TODO


### 4.4. YOLO trên toàn ảnh


In [41]:
# TODO


### 4.5. Kết hợp ROI với YOLO


In [42]:
# TODO


### 4.6. Ghép xe với ô đỗ và suy luận trạng thái


In [43]:
# TODO


## 5. Pipeline và trực quan hóa


In [44]:
# TODO


<!-- TODO -->


## 6. Đánh giá và thực nghiệm


### 6.1. Chỉ số và bộ đánh giá chung

Ghép theo `(sample_id, space_id)`; có xe là lớp dương. Ô thiếu/chưa xác định làm giảm coverage và tính sai trong accuracy tổng. Recall/F1 chính phạt ô có xe chưa xác định; `covered_only` chỉ dùng để chẩn đoán. Báo tỉ lệ báo trống sai, MAE đếm ô trống và thời gian trung bình trên ảnh gốc (gồm mọi tile).

Cell dưới kiểm chứng bằng ví dụ nhân tạo tính tay, không phải kết quả mô hình. Khi có dự đoán, điền đường dẫn JSONL; ngưỡng chỉ chọn trên validation rồi giữ nguyên khi test. Xem `docs/evaluation.md` trong repo để biết schema và công thức.


In [ ]:
# Kiểm chứng độc lập, không dùng làm kết quả thực nghiệm.
print("Evaluator self-check (synthetic):")
print(json.dumps(parking.evaluator_self_check(), indent=2))

# Điền sau khi pipeline đã xuất dự đoán cho toàn bộ split tương ứng.
EVAL_SPLIT = "val"  # Chỉ chuyển sang test sau khi đã khóa tham số.
METHOD_NAME = "method_name"
PREDICTIONS_PATH = ""  # JSONL: sample_id, space_id, occupied_score hoặc state.
TIMINGS_PATH = ""      # JSONL: thời gian trên từng ảnh gốc; để trống nếu chưa đo.
EVALUATION_THRESHOLD = 0.5

if PREDICTIONS_PATH:
    eval_frames = project_data["protocols"][PROTOCOL][EVAL_SPLIT]
    eval_metadata = dict(
        dataset_version=PROJECT_DATASET_VERSION, protocol=PROTOCOL, split=EVAL_SPLIT,
        split_lock_sha256=hashlib.sha256((PROJECT_SPLIT_DIR / "split_lock.json").read_bytes()).hexdigest(),
        predictions_sha256=hashlib.sha256(Path(PREDICTIONS_PATH).read_bytes()).hexdigest(),
        module_sha256=provenance["sha256"],
    )
    if TIMINGS_PATH:
        eval_metadata["timings_sha256"] = hashlib.sha256(Path(TIMINGS_PATH).read_bytes()).hexdigest()
    evaluation_report = parking.evaluate_occupancy(
        eval_frames, PREDICTIONS_PATH, timings=TIMINGS_PATH or None,
        threshold=EVALUATION_THRESHOLD, missing_policy="unknown",
        method=METHOD_NAME, metadata=eval_metadata,
    )
    evaluation_dir = WORK_DIR / "outputs/evaluation" / PROTOCOL / METHOD_NAME / EVAL_SPLIT
    parking.export_evaluation(evaluation_report, evaluation_dir)
    print(json.dumps(evaluation_report["overall"], indent=2))
    print("Saved:", evaluation_dir)
else:
    print("Evaluator đã sẵn sàng. Điền PREDICTIONS_PATH khi có dự đoán thật; chưa đánh giá mô hình.")


### 6.2. Validation và lựa chọn tham số


In [46]:
# TODO


### 6.3. Kiểm thử trên tập test


In [47]:
# TODO


## 7. Kết quả và thảo luận


<!-- TODO -->


## 8. Kết luận


<!-- TODO -->


## 9. Hướng dẫn tái lập


<!-- TODO -->


In [48]:
# TODO
